# DATE experiments on Kaggle (2×T4, two stages)
**Settings:** Accelerator = GPU T4 x2 · Internet = On.
**Inputs to attach:** (1) a Kaggle dataset containing `date_pipeline.py` and `dataset.csv`; (2) the competition `social-media-extremism-detection-challenge` (optional – enables the label-version analysis via its `train.csv`); (3) for **Stage B only**: the output of the Stage A run (Add Input → Notebook Output), or a dataset containing its `runs/` folder.

Set `STAGE = "A"` in Cell 3 for the first session (~4 h wall on 2 GPUs: core + 9 other encoders + learning curves + augmentation operators, seed 42). Then start a second session with `STAGE = "B"` (~4.5 h: seeds 43–46 for the core grid).

In [ ]:
# Cell 1: GPUs + locate files
import glob, os, subprocess, shutil, time
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
SCRIPT = glob.glob("/kaggle/input/**/date_pipeline.py", recursive=True)
CSV = glob.glob("/kaggle/input/**/dataset.csv", recursive=True)
COMP = [p for p in glob.glob("/kaggle/input/**/train.csv", recursive=True)]
PREV = sorted(glob.glob("/kaggle/input/**/runs/date_full_s42/fold0.npz", recursive=True))
print("script:", SCRIPT); print("csv:", CSV); print("competition train.csv:", COMP); print("previous stage output:", PREV)
assert SCRIPT and CSV, "attach a dataset containing date_pipeline.py and dataset.csv"


In [ ]:
# Cell 2: copy script, point to data, restore previous stage (Stage B only)
STAGE = "A"          # <<< "A" first session, "B" second session
os.chdir("/kaggle/working")
shutil.copy(SCRIPT[0], "/kaggle/working/date_pipeline.py")
os.environ["DATA_CSV"] = CSV[0]
if COMP: os.environ["COMP_TRAIN"] = COMP[0]
os.makedirs("/kaggle/working/outputs", exist_ok=True)
if STAGE == "B":
    assert PREV, "Stage B needs the Stage A output attached as an input"
    base = os.path.dirname(os.path.dirname(os.path.dirname(PREV[0])))      # .../outputs
    for sub in ("runs", "aug"):
        if os.path.isdir(f"{base}/{sub}"): shutil.copytree(f"{base}/{sub}", f"/kaggle/working/outputs/{sub}", dirs_exist_ok=True)
    print("restored:", os.listdir("/kaggle/working/outputs/runs")[:50])


In [ ]:
# Cell 3: launch one process per GPU (balanced by estimated time). Safe to re-run: finished folds are skipped.
PLAN = {
 "A": {"seeds": "42", "gpu0": "date_full,no_cw,no_cosine,date_grouped,deberta_v3,bertweet_large,curve",
                      "gpu1": "no_aug,roberta_base,bert_base,distilbert,xlm_roberta,twitter_roberta,twitter_roberta_hate,twhin_bert,toxic_bert,augops"},
 "B": {"seeds": "43,44,45,46", "gpu0": "date_full,no_cw,no_cosine", "gpu1": "no_aug,roberta_base,date_grouped"},
}[STAGE]
def launch(gpu, runs, seeds, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), RUN=runs, SEEDS=seeds, PYTHONUNBUFFERED="1")
    return subprocess.Popen(["python", "date_pipeline.py"], env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT, cwd="/kaggle/working")
p0 = launch(0, PLAN["gpu0"], PLAN["seeds"], "gpu0.log"); time.sleep(120)      # stagger: GPU0 builds the shared augmentation cache first
p1 = launch(1, PLAN["gpu1"], PLAN["seeds"], "gpu1.log")
p2 = None   # Stage B only: the DeBERTa-v3 seed-42 rerun (fp32 weights fix) starts on GPU 1 as soon as p1 finishes
print("stage", STAGE, "launched")


In [ ]:
# Cell 4: monitor until everything finishes (prints the shared log every 5 min; safe to interrupt and re-run)
def tail(path, n=8):
    try: return "".join(open(path, encoding="utf-8", errors="ignore").readlines()[-n:])
    except Exception as e: return str(e)
def running(p): return p is not None and p.poll() is None
while running(p0) or running(p1) or running(p2) or (STAGE == "B" and p2 is None):
    if STAGE == "B" and p2 is None and p1.poll() is not None:
        p2 = launch(1, "deberta_v3", "42", "gpu1b.log"); print("DeBERTa-v3 (seed 42) rerun launched on GPU 1")
    print(time.strftime("%H:%M:%S"), "| running:", running(p0), running(p1), running(p2))
    print(tail("/kaggle/working/outputs/run.log")); time.sleep(300)
print("exit codes:", p0.returncode, p1.returncode, None if p2 is None else p2.returncode)
if p0.returncode or p1.returncode or (p2 is not None and p2.returncode): print(tail("gpu0.log", 30)); print(tail("gpu1.log", 30))
print([l for l in open("/kaggle/working/outputs/run.log", encoding="utf-8", errors="ignore") if "FAILED" in l])


In [ ]:
# Cell 5: analysis-only pass over everything (no training): tables, figures, exports, zip
SEEDS_ALL = "42" if STAGE == "A" else "42,43,44,45,46"
env = dict(os.environ, ANALYZE_ONLY="1", SEEDS=SEEDS_ALL, RUN="all")
env.pop("CUDA_VISIBLE_DEVICES", None)
r = subprocess.run(["python", "date_pipeline.py"], cwd="/kaggle/working", env=env, capture_output=True, text=True)
print(r.stdout[-5000:]); print(r.stderr[-2500:])


In [ ]:
# Cell 6: peek at the key tables and the download file
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60)
T = "/kaggle/working/outputs/tables/"
for f in ["summary_all_configs", "calibration_single_vs_ensemble", "multiseed_summary", "config_comparisons", "external_transfer", "robustness_and_fairness_probes"]:
    if os.path.exists(T + f + ".csv"): print("=====", f); print(pd.read_csv(T + f + ".csv").head(40).T if f == "summary_all_configs" else pd.read_csv(T + f + ".csv").head(40))
print(os.path.getsize("/kaggle/working/outputs_bundle.zip") / 1e6, "MB -> download /kaggle/working/outputs_bundle.zip")
